# Hidrostatska sila na ravnu plohu — predvidi, izračunaj, provjeri

**Poglavlje U05: hidrostatske sile na plohe**

Tlak ćemo integrirati po pravokutnoj plohi i neovisno usporediti s izrazima
preko dubine težišta. Kut $\alpha$ ovdje je izričito mjeren **od vertikale**;
gornji rub nalazi se na dubini $h_t$.


## 1. Predvidi

1. Leži li centar tlaka bliže gornjem ili donjem rubu plohe?
2. Kako se razlika između centra tlaka i težišta mijenja kada cijelu plohu
   spustimo mnogo dublje?
3. Što očekuješ za vodoravnu plohu ($\alpha=90^\circ$ od vertikale)?

Nacrtaj os $y$ niz plohu prije pokretanja računa.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
RHO, G = 998.0, 9.81

def analiticka_ploha(h_t, b, L, alpha_deg):
    c = np.cos(np.radians(alpha_deg))
    F = RHO*G*L*(h_t*b + 0.5*c*b**2)
    M_gornji = RHO*G*L*(0.5*h_t*b**2 + (c/3.0)*b**3)
    y_cp = M_gornji/F
    return F, y_cp, h_t + c*y_cp

h_t, b, L, alpha = 0.60, 1.80, 1.20, 35.0
F_ref, ycp_ref, hcp_ref = analiticka_ploha(h_t, b, L, alpha)
print(f"F = {F_ref/1000:.3f} kN")
print(f"y_CP = {ycp_ref:.4f} m od gornjeg ruba; h_CP = {hcp_ref:.4f} m")


## 2. Izračunaj — integracija raspodjele tlaka

Duž plohe je $h(y)=h_t+y\cos\alpha$ i $p(y)=\rho gh(y)$. Numerički računamo

$$F=\int_0^b p(y)L\,dy,\qquad
M_t=\int_0^b y\,p(y)L\,dy,\qquad y_{CP}=M_t/F.$$

Trapezno pravilo točno integrira linearnu raspodjelu sile, ali momentni
integrand je kvadratičan, pa se pogreška centra tlaka smanjuje s mrežom.


In [ ]:
def trapz_local(y, x):
    return np.sum(0.5*(y[:-1]+y[1:]) * np.diff(x))

def numericka_ploha(h_t, b, L, alpha_deg, n):
    y = np.linspace(0.0, b, n+1)
    h = h_t + y*np.cos(np.radians(alpha_deg))
    p = RHO*G*h
    F = trapz_local(p*L, y)
    M = trapz_local(y*p*L, y)
    return F, M/F, y, p

n_mreza = np.array([4, 8, 16, 32, 64, 128])
rezultati = [numericka_ploha(h_t, b, L, alpha, int(n)) for n in n_mreza]
F_num = np.array([r[0] for r in rezultati])
ycp_num = np.array([r[1] for r in rezultati])
err_y = np.abs(ycp_num-ycp_ref)
red = np.log2(err_y[:-1]/err_y[1:])
print("n   F [kN]      y_CP [m]    |pogreška y_CP| [m]")
for n, fn, yn, en in zip(n_mreza, F_num, ycp_num, err_y):
    print(f"{n:3d} {fn/1000:10.5f} {yn:12.7f} {en:18.3e}")

_, _, y_plot, p_plot = numericka_ploha(h_t, b, L, alpha, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.2, 3.8))
ax1.plot(p_plot/1000, y_plot, color="#1565c0")
ax1.axhline(ycp_ref, color="#c62828", ls="--", label="centar tlaka")
ax1.invert_yaxis()
ax1.set(xlabel="manometarski tlak (kPa)", ylabel="y niz plohu (m)")
ax1.legend()
ax1.grid(ls=":", alpha=0.5)
ax2.loglog(b/n_mreza, err_y, "o-")
ax2.set(xlabel=r"korak $\Delta y$ (m)", ylabel=r"$|y_{CP}-y_{CP,ref}|$ (m)",
        title="Konvergencija centra tlaka")
ax2.grid(ls=":", which="both", alpha=0.5)
fig.tight_layout()
plt.show()


## 3. Provjeri — sila, moment i granični slučaj

Sila i moment uspoređuju se zasebno s analitičkim integralima. Vodoravna
ploha ima uniforman tlak, pa centar tlaka mora biti u sredini plohe.


In [ ]:
F_fin, ycp_fin, _, _ = numericka_ploha(h_t, b, L, alpha, 512)
M_fin = F_fin*ycp_fin
M_ref = F_ref*ycp_ref
_, ycp_horizontalno, _ = analiticka_ploha(h_t, b, L, 90.0)

print(f"opaženi red zadnja tri refiniranja: {red[-3:]}")
assert np.isclose(F_fin, F_ref, rtol=2e-12)
assert abs(M_fin-M_ref)/M_ref < 2e-6
assert np.isclose(ycp_horizontalno, b/2, rtol=1e-12)
assert np.all((red[-3:] > 1.99) & (red[-3:] < 2.01))
print("PASS: sila, moment, vodoravna ploha i drugi red konvergencije.")


## Granica modela

Račun daje rezultantu poznatog hidrostatičkog opterećenja. Ne uključuje
tlak s druge strane plohe, dinamiku fluida, deformaciju ni provjeru čvrstoće.
Tvrdnja o položaju centra tlaka mora se vezati uz zadanu raspodjelu tlaka;
nije univerzalna za svaku kombinaciju referentnih tlakova.
